<a href="https://colab.research.google.com/github/AshokGit544/ai-clinical-triage-assistant/blob/main/AI_Clinical_Triage_Assistant_(LLM_%2B_RAG).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# INSTALLING LIBRARIES
#
!pip install sentence-transformers faiss-cpu pandas requests --quiet


# IMPORT LIBRARIES

import pandas as pd
import numpy as np
import faiss
import requests
from sentence_transformers import SentenceTransformer
from getpass import getpass


# OPENROUTER API

OPENROUTER_API_KEY = getpass("Enter OpenRouter API Key: ")

MODEL_NAME = "nvidia/nemotron-3-nano-30b-a3b:free"


# CALL LLM FUNCTION

def call_llm(prompt):

    response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": MODEL_NAME,
            "messages":[
                {"role":"system","content":"You are a clinical triage AI assistant for primary care."},
                {"role":"user","content":prompt}
            ]
        }
    )

    result = response.json()

    if "choices" in result:
        return result["choices"][0]["message"]["content"]

    return result


# SAMPLE MEDICAL KNOWLEDGE BASE

data = [
["COVID-19","fever cough fatigue shortness_of_breath","respiratory infection"],
["Flu","fever headache fatigue cough","viral infection"],
["Pneumonia","fever cough chest pain breathing difficulty","lung infection"],
["Diabetes complication","fatigue thirst frequent urination","metabolic disorder"],
["Heart attack","chest pain shortness_of_breath nausea","cardiac emergency"]
]

df = pd.DataFrame(data, columns=["Condition","Symptoms","Category"])


# CREATE EMBEDDINGS

print("Loading embedding model...")

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(df["Symptoms"].tolist())
embeddings = np.array(embeddings).astype("float32")


# CREATE VECTOR DATABASE

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("Vector database created")


# RETRIEVE RELEVANT CONDITIONS
def retrieve_conditions(query,k=3):

    q = model.encode([query])
    q = np.array(q).astype("float32")

    distances, indices = index.search(q,k)

    results = df.iloc[indices[0]]

    return results


# TRIAGE PIPELINE

def clinical_triage(symptoms, age, history):

    retrieved = retrieve_conditions(symptoms)

    context = retrieved.to_string()

    prompt = f"""

You are a clinical AI triage assistant.

Patient Information
Age: {age}
Medical History: {history}
Symptoms: {symptoms}

Medical Knowledge Base
{context}

Task:
1. Determine urgency level (Low / Medium / High)
2. Suggest possible conditions
3. Recommend next step

Return response in this format:

Urgency Level:
Possible Conditions:
Recommended Action:

"""

    answer = call_llm(prompt)

    return answer

# USER INPUT
symptoms = input("Enter patient symptoms: ")
age = input("Enter patient age: ")
history = input("Enter medical history: ")

# RUN TRIAGE

result = clinical_triage(symptoms,age,history)

print("\n===== AI CLINICAL TRIAGE RESULT =====\n")

print(result)